# <font color = "#009d71">**Trabalho PI - Criação de coleções com PyMongo**
### **Pontifícia Universidade Católica de Campinas**
### **Prof. Felipe Cavalaro**
### **Integrantes**
- Francisco da Silva Bueno Junior - 25008051
- Isabel Baungartner - 25001436
- Italo Fraga Botelho - 25894064
- Tiago Noda Von Zuben - 25018493



---



In [1]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 22.3 MB/s eta 0:00:00


In [2]:
import pymongo
import pandas as pd

In [6]:
# conexão com MongoDB
client = pymongo.MongoClient(
"mongodb+srv://isabelbaungartner_db_user:<PASSWORD>@cluster0.a6z6ys0.mongodb.net/?appName=Cluster0")

# banco de dados
db = client["siest_db"]



---



In [ ]:
# Limpeza de segurança das coleções agregadas antigas
colecoes_para_limpar = [
    "agg_casos_por_ano",
    "agg_casos_por_mes",
    "agg_casos_por_hospital",
    "agg_resumo_clima",
    "agg_risco_inundacao",
    "agg_vulnerabilidade_habitacional",
    "agg_casos_clima_por_mes",
    "agg_casos_por_faixa_etaria",
    "agg_casos_por_sexo",
    "agg_clima_por_mes",
    "agg_hospitalizacao",
    "agg_mapa_casos",
    "agg_resumo_casos",
    "agg_letalidade_doenca"

]

for col in colecoes_para_limpar:
  db[col].drop()

print("Coleções antigas deletadas. O banco está limpo para receber as novas agregações!")

Coleções antigas deletadas. O banco está limpo para receber as novas agregações!


#**CRIANDO COLEÇÕES**

In [4]:
#função para criar agregações

def salvar_agregacao(nome_colecao_origem, pipeline, nome_colecao_destino):
    resultado = list(db[nome_colecao_origem].aggregate(pipeline))

    db[nome_colecao_destino].delete_many({})

    if resultado:
        db[nome_colecao_destino].insert_many(resultado)

    print(f"{nome_colecao_destino}: {len(resultado)} documentos salvos")

###<font color = "#5ccda7">**Total de casos por ano**


```
Para a criação de um gráfico que mostra uma linha temporal dos casos das doenças por ano.
```



In [ ]:
pipeline_casos_por_ano = [
    {
        "$group": {
            "_id": {
                "ano": {"$year": "$DT_NOTIFIC"},
                "doenca": "$NOME_DOENCA"
            },
            "total_casos": {"$sum": 1}
        }
    },
    {"$sort": {"_id.ano": 1}},
    {
        "$project": {
            "_id": 0, "ano": "$_id.ano", "doenca": "$_id.doenca", "total_casos": 1
        }
    }
]

salvar_agregacao(
    "casos_geolocalizados",
    pipeline_casos_por_ano,
    "agg_casos_por_ano"
)

###<font color = "#5ccda7">**Total de casos por mês**

In [ ]:
pipeline_casos_por_mes = [
    {
        "$group": {
            "_id": {
                "ano": {"$year": "$DT_NOTIFIC"},
                "mes": {"$month": "$DT_NOTIFIC"},
                "doenca": "$NOME_DOENCA"
            },
            "total_casos": {"$sum": 1}
        }
    },
    {"$sort": {"_id.ano": 1, "_id.mes": 1}},
    {
        "$project": {
            "_id": 0,
            "ano": "$_id.ano",
            "mes": "$_id.mes",
            "doenca": "$_id.doenca",
            "total_casos": 1
        }
    }
]

salvar_agregacao(
    "casos_geolocalizados",
    pipeline_casos_por_mes,
    "agg_casos_por_mes"
)

agg_casos_por_mes: 652 documentos salvos


###<font color = "#5ccda7">**Casos por hospital/unidade**


```
Gráfico: ranking de hospitais com mais notificações.
```



In [ ]:
pipeline_casos_por_hospital = [
    {
        "$group": {
            "_id": {
                "hospital": "$NO_FANTASIA",
                "doenca": "$NOME_DOENCA"
            },
            "total_casos": {"$sum": 1},
            "hospitalizacoes": {
                "$sum": {
                    "$cond": [{"$in": ["$HOSPITALIZ", ["1", 1, "Sim", "SIM"]]}, 1, 0]
                }
            }
        }
    },
    {"$sort": {"total_casos": -1}},
    {
        "$project": {
            "_id": 0, "hospital": "$_id.hospital", "doenca": "$_id.doenca",
            "total_casos": 1, "hospitalizacoes": 1
        }
    }
]
salvar_agregacao("casos_geolocalizados", pipeline_casos_por_hospital, "agg_casos_por_hospital")

agg_casos_por_hospital: 614 documentos salvos


###<font color = "#5ccda7">**Casos por sexo**

In [ ]:
pipeline_casos_por_sexo = [
    {
        "$group": {
            "_id": {
                "sexo": "$CS_SEXO",
                "doenca": "$NOME_DOENCA"
            },
            "total_casos": {"$sum": 1}
        }
    },
    {"$sort": {"total_casos": -1}},
    {
        "$project": {"_id": 0, "sexo": "$_id.sexo", "doenca": "$_id.doenca", "total_casos": 1}
    }
]
salvar_agregacao("casos_geolocalizados", pipeline_casos_por_sexo, "agg_casos_por_sexo")

agg_casos_por_sexo: 13 documentos salvos


###<font color = "#5ccda7">**Casos por faixa etária**

In [ ]:
pipeline_faixa_etaria = [
    {
        "$addFields": {
            "idade_real": {
                "$let": {
                    "vars": {
                        "idade_num": {
                            "$convert": {
                                "input": "$NU_IDADE_N",
                                "to": "double",
                                "onError": None,
                                "onNull": None
                            }
                        }
                    },
                    "in": {
                        "$cond": [
                            {"$eq": ["$$idade_num", None]},
                            None,
                            {"$cond": [
                                {"$gte": ["$$idade_num", 4000]},
                                {"$mod": ["$$idade_num", 1000]},
                                0
                            ]}
                        ]
                    }
                }
            }
        }
    },

    {
        "$addFields": {
            "faixa_etaria": {
                "$switch": {
                    "branches": [
                        {"case": {"$eq": ["$idade_real", None]}, "then": "Não Informado"},
                        {"case": {"$lte": ["$idade_real", 4]}, "then": "00 a 04 anos"},   # Primeira Infância (Alta vulnerabilidade)
                        {"case": {"$lte": ["$idade_real", 14]}, "then": "05 a 14 anos"},  # Crianças e Pré-adolescentes
                        {"case": {"$lte": ["$idade_real", 29]}, "then": "15 a 29 anos"},  # Jovens
                        {"case": {"$lte": ["$idade_real", 39]}, "then": "30 a 39 anos"},  # Adultos I
                        {"case": {"$lte": ["$idade_real", 59]}, "then": "40 a 59 anos"},  # Adultos II
                        {"case": {"$lte": ["$idade_real", 79]}, "then": "60 a 79 anos"}   # Idosos
                    ],
                    "default": "80+ anos" # Longevidade (Altíssima letalidade)
                }
            }
        }
    },

    {
        "$group": {
            "_id": {
                "doenca": "$NOME_DOENCA",
                "faixa_etaria": "$faixa_etaria"
            },
            "total_casos": {"$sum": 1}
        }
    },

    {
        "$project": {
            "_id": 0,
            "doenca": "$_id.doenca",
            "faixa_etaria": "$_id.faixa_etaria",
            "total_casos": 1
        }
    },

    {
        "$sort": {
            "faixa_etaria": 1
        }
    }
]

salvar_agregacao(
    "casos_geolocalizados",
    pipeline_faixa_etaria,
    "agg_casos_por_faixa_etaria"
)

agg_casos_por_faixa_etaria: 35 documentos salvos


###<font color = "#5ccda7">**Casos hospitalizados x não hospitalizados**

In [ ]:
pipeline_hospitalizacao = [
    {
        "$group": {
            "_id": {
                "hospitalizado": "$HOSPITALIZ",
                "doenca": "$NOME_DOENCA"
            },
            "total": {"$sum": 1}
        }
    },
    {
        "$project": {"_id": 0, "doenca": "$_id.doenca", "hospitalizado": "$_id.hospitalizado", "total": 1}
    }
]
salvar_agregacao("casos_geolocalizados", pipeline_hospitalizacao, "agg_hospitalizacao")

agg_hospitalizacao: 9 documentos salvos


###<font color = "#5ccda7">**média climática por mês**

In [ ]:
pipeline_clima_por_mes = [
    {
        "$group": {
            "_id": {
                "ano": {"$year": "$DT_MEDICAO"},
                "mes": {"$month": "$DT_MEDICAO"}
            },
            "precipitacao_total": {"$sum": "$PRECIP_TOTAL"},
            "temperatura_media": {"$avg": "$TEMP_MED"},
            "umidade_media": {"$avg": "$UMIDADE_MED"},
            "temperatura_maxima": {"$max": "$TEMP_MAX"}
        }
    },
    {"$sort": {"_id.ano": 1, "_id.mes": 1}},
    {
        "$project": {
            "_id": 0,
            "ano": "$_id.ano",
            "mes": "$_id.mes",
            "precipitacao_total": {"$round": ["$precipitacao_total", 2]},
            "temperatura_media": {"$round": ["$temperatura_media", 2]},
            "umidade_media": {"$round": ["$umidade_media", 2]},
            "temperatura_maxima": {"$round": ["$temperatura_maxima", 2]}
        }
    }
]

salvar_agregacao(
    "clima",
    pipeline_clima_por_mes,
    "agg_clima_por_mes"
)

agg_clima_por_mes: 149 documentos salvos


###<font color = "#5ccda7">**Casos por localização para mapa**

In [ ]:
pipeline_mapa_casos = [
    {
        "$group": {
            "_id": {
                "hospital": "$NO_FANTASIA",
                "latitude": "$LATITUDE",
                "longitude": "$LONGITUDE",
                "doenca": "$NOME_DOENCA"
            },
            "total_casos": {"$sum": 1}
        }
    },
    {
        "$project": {
            "_id": 0, "hospital": "$_id.hospital", "doenca": "$_id.doenca",
            "latitude": "$_id.latitude", "longitude": "$_id.longitude", "total_casos": 1
        }
    }
]
salvar_agregacao("casos_geolocalizados", pipeline_mapa_casos, "agg_mapa_casos")

agg_mapa_casos: 620 documentos salvos


###<font color = "#5ccda7">**Classes de risco de inundação**

In [ ]:
pipeline_risco_inundacao = [
    {
        "$group": {
            "_id": "$properties.CLASSE",
            "total_areas": {"$sum": 1}
        }
    },
    {"$sort": {"total_areas": -1}},
    {
        "$project": {"_id": 0, "classe_risco": "$_id", "total_areas": 1}
    }
]

salvar_agregacao(
    "risco_inundacao",
    pipeline_risco_inundacao,
    "agg_risco_inundacao"
)

agg_risco_inundacao: 3 documentos salvos


###<font color = "#5ccda7">**Áreas de vulnerabilidade habitacional**

In [ ]:
pipeline_vulnerabilidade = [
    {
        "$group": {
            "_id": "$properties.NOME_AREA",
            "total_registros": {"$sum": 1}
        }
    },
    {"$sort": {"total_registros": -1}},
    {
        "$project": {"_id": 0, "area": "$_id", "total_registros": 1}
    }
]

salvar_agregacao(
    "vulnerabilidade_habitacional",
    pipeline_vulnerabilidade,
    "agg_vulnerabilidade_habitacional"
)

agg_vulnerabilidade_habitacional: 531 documentos salvos


###<font color = "#5ccda7">**Dashboard principal: resumo geral**

In [7]:
pipeline_resumo_casos = [
    {
        "$group": {
            "_id": None,
            "total_casos": {"$sum": 1},

            # Tratamento avançado de Idade do SINAN
            "media_idade": {
                "$avg": {
                    "$let": {
                        "vars": {
                            # Tenta converter para inteiro. Se der erro (ex: string vazia), vira nulo
                            "idade_num": {
                                "$convert": {
                                    "input": "$NU_IDADE_N",
                                    "to": "int",
                                    "onError": None,
                                    "onNull": None
                                }
                            }
                        },
                        "in": {
                            # Só faz o cálculo se a conversão deu certo
                            "$cond": [
                                {"$eq": ["$$idade_num", None]},
                                None, # Ignora na média

                                # Lógica SINAN: 4000+ significa Anos. Menos que 4000 é meses/dias.
                                {"$cond": [
                                    {"$gte": ["$$idade_num", 4000]},
                                    {"$mod": ["$$idade_num", 1000]}, # Pega o resto da divisão por 1000 (ex: 4025 vira 25)
                                    0 # Bebês (meses ou dias) entram como 0 anos
                                ]}
                            ]
                        }
                    }
                }
            },

            "total_hospitalizados": {
                "$sum": {
                    "$cond": [
                        {"$in": ["$HOSPITALIZ", ["1", 1, "Sim", "SIM", 1.0]]},
                        1,
                        0
                    ]
                }
            },
            "total_unidades": {"$addToSet": "$NO_FANTASIA"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "total_casos": 1,
            "media_idade": {"$round": ["$media_idade", 2]},
            "total_hospitalizados": 1,
            "total_unidades": {"$size": "$total_unidades"}
        }
    }
]

# Roda e salva a agregação
salvar_agregacao(
    "casos_geolocalizados",
    pipeline_resumo_casos,
    "agg_resumo_casos"
)

agg_resumo_casos: 1 documentos salvos


###<font color = "#5ccda7">**Resumo climático**

In [ ]:
pipeline_resumo_clima = [
    {
        "$group": {
            "_id": None,
            "media_temperatura": {"$avg": "$TEMP_MED"},
            "maior_temperatura": {"$max": "$TEMP_MAX"}, # Corrigido para as novas vars
            "menor_temperatura": {"$min": "$TEMP_MIN"}, # Corrigido para as novas vars
            "umidade_media": {"$avg": "$UMIDADE_MED"}, # Nova Var
            "total_precipitacao": {"$sum": "$PRECIP_TOTAL"},
            "maior_chuva_dia": {"$max": "$PRECIP_TOTAL"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "media_temperatura": {"$round": ["$media_temperatura", 2]},
            "maior_temperatura": 1, "menor_temperatura": 1,
            "umidade_media": {"$round": ["$umidade_media", 2]},
            "total_precipitacao": {"$round": ["$total_precipitacao", 2]},
            "maior_chuva_dia": 1
        }
    }
]
salvar_agregacao("clima", pipeline_resumo_clima, "agg_resumo_clima")

agg_resumo_clima: 1 documentos salvos


###<font color = "#5ccda7">**Relação casos + clima por mês**

In [ ]:
pipeline_casos_clima_mes = [
    {
        "$group": {
            "_id": {
                "ano": {"$year": "$DT_NOTIFIC"},
                "mes": {"$month": "$DT_NOTIFIC"},
                "doenca": "$NOME_DOENCA"
            },
            "total_casos": {"$sum": 1}
        }
    },
    {
        "$lookup": {
            "from": "agg_clima_por_mes", # Certifique-se que já rodou aquele que corrigimos na mensagem anterior
            "let": {"ano_caso": "$_id.ano", "mes_caso": "$_id.mes"},
            "pipeline": [
                {
                    "$match": {
                        "$expr": {
                            "$and": [{"$eq": ["$ano", "$$ano_caso"]}, {"$eq": ["$mes", "$$mes_caso"]}]
                        }
                    }
                }
            ],
            "as": "clima"
        }
    },
    {"$unwind": "$clima"},
    {
        "$project": {
            "_id": 0, "ano": "$_id.ano", "mes": "$_id.mes", "doenca": "$_id.doenca", "total_casos": 1,
            "precipitacao_total": "$clima.precipitacao_total",
            "temperatura_media": "$clima.temperatura_media",
            "temperatura_maxima": "$clima.temperatura_maxima",
            "umidade_media": "$clima.umidade_media"
        }
    },
    {"$sort": {"ano": 1, "mes": 1}}
]
salvar_agregacao("casos_geolocalizados", pipeline_casos_clima_mes, "agg_casos_clima_por_mes")

agg_casos_clima_por_mes: 651 documentos salvos


###<font color = "#5ccda7">**Taxa de Letalidade**

In [ ]:
pipeline_letalidade_doenca = [
    {
        "$group": {
            "_id": {
                "doenca": "$NOME_DOENCA",
                "evolucao": "$EVOLUCAO"
            },
            "total": {"$sum": 1}
        }
    },
    {
        "$project": {
            "_id": 0, "doenca": "$_id.doenca", "evolucao": "$_id.evolucao", "total": 1
        }
    }
]
salvar_agregacao("casos_geolocalizados", pipeline_letalidade_doenca, "agg_letalidade_doenca")

agg_letalidade_doenca: 18 documentos salvos
